# θ-only adaptation from CSGM (SR ×4, development images)

**Question.** Take finished CSGM reconstructions, **freeze their noise ε**, and optimize **only the generator weights θ** for 2,000 steps. How much does quality change from before (CSGM) to after (CSGM + θ)?

This is IADrift with the ε learning rate set to exactly zero and run twice as long as before. Notebook 02 found that ε barely matters (ε lr 0 ≈ 1e-3 over 1,000 steps). This notebook isolates the θ part cleanly and shows whether more steps continue the same trend.

**Development images only, correct (oracle) label only.** No test image is used.

## What exactly was done

**Images.** The 4 development images that already have cached CSGM solutions: 8117 (streetcar), 3939 (bonnet), 403 (crane), 347 (spoonbill). `check_dev` refuses anything in the test split.

**Before = "CSGM best".** For each image we load the 2 cached CSGM restarts from `csgm_dev_5000steps_lr0.02/` (frozen generator, Adam lr 0.02 on ε, 5,000 steps) and keep the restart with the **lowest final LR loss** (never HR). Its ε and `noise_labels` are the starting point.

**After = "CSGM + θ".**
- θ starts at the pretrained weights (a fresh copy for every run).
- ε, `noise_labels` and the class label are **held fixed**: they are constants in the loss, not optimized at all (no ε optimizer exists in this step).
- Loss: $\|y - A f_\theta(\epsilon_{CSGM})\|^2$, the same LR measurement loss as CSGM and IADrift. HR is never used by the optimization.
- Optimizer: Adam on all 241 parameter leaves (134M parameters), **2,000 steps**, jitted step with donated buffers. Two learning rates, each run separately from the same start:
  - **θ lr 1e-5**: the setting used in all earlier IADrift runs.
  - **θ lr 1e-6**: the gentlest setting in notebook 02 (smallest PSNR loss after 1,000 steps).

**Monitoring.** Every 50 steps (and at step 0) we record LR loss, HR PSNR and band SNR through the compiled generator. These curves are evaluation only and are slightly offset from eager values because of bf16 compilation differences.

**Final metrics** for both before and after come from the **eager** generator path, like every saved CSGM number. We also report the HR-best step along the curve, but that uses HR, so it's an *oracle* diagnostic, not a usable stopping rule.

**Metrics.** HR PSNR (dB, data range 2; unclipped main, clipped also), LR loss ‖y − A x‖², band SNR low / mid / high / very high (`freq_metrics.py`; 0 dB = band left empty, negative = worse than empty). Reference rows: bicubic and bicubic + BP (the strongest prior-free baseline from notebook 04).

**Visualization.** Per image, side by side: original HR, bicubic, **CSGM best**, **CSGM + θ (1e-6)**, **CSGM + θ (1e-5)**, plus a zoomed crop of each to show detail.

**Caching.** Each run is saved to `~/work/datasets/ImageNetV2/theta_only_dev_v1/`; re-running loads it instead of recomputing.

**Caveats.** 4 images; no significance tests are meaningful at n = 4, so per-image values are shown.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # must be set before JAX is imported (θ-gradients)
SCRIPTS = Path.home() / "work" / "IADrift" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import io
import json
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from jax.scipy.sparse.linalg import cg

# Project helpers (importing them loads the generator).
from freq_metrics import BANDS, band_snr_db
from run_sr4_csgm_test import FROZEN_CONFIG, load_split, optax
from run_sr4_iadrift_dev import CSGM_CACHE, check_dev
from run_sr4_iadrift_smoke import (
    A_sr4, DATA_DIR, atomic_write_bytes, bicubic_up, class_name, eager_metrics,
    generate_jit, iadrift_loss, jax, jnp, load_hr256, load_index, params, psnr,
)

BAND_NAMES = [name for name, *_ in BANDS]
%matplotlib inline
print("Devices:", jax.devices())

## θ-only optimizer step, back-projection and metrics

`make_theta_step` is `make_joint_step` from `run_sr4_iadrift_smoke.py` with the ε half removed: the gradient is taken with respect to θ only, and ε is passed in as a constant. Back-projection is only used for the bicubic + BP reference row.

In [ ]:
def make_theta_step(lr_theta):
    opt = optax.adam(lr_theta)

    def step(theta, state, eps, y, label, noise_labels):
        loss, g = jax.value_and_grad(iadrift_loss, argnums=0)(theta, eps, y, label, noise_labels)
        updates, state = opt.update(g, state, theta)
        return optax.apply_updates(theta, updates), state, loss

    # θ and its optimizer state are donated (updated in place); ε is a constant input.
    return opt, jax.jit(step, donate_argnums=(0, 1))


_A_transpose = jax.linear_transpose(A_sr4, jnp.zeros((1, 256, 256, 3), jnp.float32))


@jax.jit
def _backproject(x, y):
    def AT(v):
        return _A_transpose(v)[0]
    u, _ = cg(lambda v: A_sr4(AT(v)), y - A_sr4(x), tol=1e-6, maxiter=500)
    return x + AT(u)


def lr_loss(x, y):
    return float(jnp.sum((A_sr4(jnp.asarray(x, jnp.float32)[None]) - y) ** 2))


def evaluate(x, hr, y):
    return {"psnr": psnr(hr, x), "psnr_clipped": psnr(hr, np.clip(x, -1, 1)),
            "lr_loss": lr_loss(x, y), "band_snr_db": band_snr_db(x, hr)}

## Load CSGM starting points and run θ-only adaptation

For each image: load both CSGM restarts, keep the lowest-LR-loss one, re-evaluate it eagerly with the pretrained θ (this is "CSGM best"), then run 2,000 θ-only steps per learning rate.

In [ ]:
DEV_IMAGES = [8117, 3939, 403, 347]
LR_THETAS = [1e-6, 1e-5]
STEPS = 2_000
EVAL_EVERY = 50
OUT_DIR = DATA_DIR / "theta_only_dev_v1"
OUT_DIR.mkdir(exist_ok=True)

index_rows = load_index()
labels = np.array([int(row["label"]) for row in index_rows])
train_indices, test_indices = load_split(labels)
check_dev(DEV_IMAGES, train_indices, test_indices)  # refuses test images
lr_all = np.load(DATA_DIR / "imageNet_x4.npy", mmap_mode="r")

images = {}
for i in DEV_IMAGES:
    label = int(labels[i])
    hr = load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = jnp.asarray(lr_all[i:i + 1], jnp.float32)
    restarts = []
    for r in range(FROZEN_CONFIG["num_restarts"]):
        with np.load(CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz") as data:
            m = json.loads(str(data["metrics_json"]))
            assert m["class_label"] == label
            restarts.append((m["final_loss"], r, data["eps"], data["noise_labels"]))
    _, restart, eps, noise_labels = min(restarts, key=lambda t: (t[0], t[1]))
    eps, noise_labels = jnp.asarray(eps), jnp.asarray(noise_labels)
    csgm, _, _ = eager_metrics(params, eps, label, noise_labels, y, hr)
    bic = bicubic_up(lr_all[i])
    bic_bp = np.asarray(_backproject(jnp.asarray(bic)[None], y)[0])
    images[i] = {"label": label, "hr": hr, "y": y, "eps": eps, "noise_labels": noise_labels, "restart": restart,
                 "outputs": {"bicubic": bic, "bicubic + BP": bic_bp, "CSGM best": csgm}}
    print(f"image {i} ({class_name(label)}): CSGM restart r{restart}, eager PSNR {psnr(hr, csgm):.2f} dB")


def run_theta_only(i, lr_theta, step_fn, opt):
    path = OUT_DIR / f"img{i:05d}_lrt{lr_theta:g}_steps{STEPS}.npz"
    if path.exists():
        with np.load(path) as data:
            return json.loads(str(data["result_json"])), data["x"]
    im = images[i]
    theta = jax.tree_util.tree_map(jnp.copy, params)  # fresh copy; the original params are never donated
    state = opt.init(theta)
    trace = {"step": [], "lr_loss": [], "psnr": [], **{b: [] for b in BAND_NAMES}}

    def record(step):
        x = np.asarray(generate_jit(theta, im["eps"], im["label"], im["noise_labels"])[0])
        m = evaluate(x, im["hr"], im["y"])
        trace["step"].append(step)
        trace["lr_loss"].append(m["lr_loss"])
        trace["psnr"].append(m["psnr"])
        for b in BAND_NAMES:
            trace[b].append(m["band_snr_db"][b])

    start = perf_counter()
    record(0)
    for step in range(1, STEPS + 1):
        theta, state, _ = step_fn(theta, state, im["eps"], im["y"], im["label"], im["noise_labels"])
        if step % EVAL_EVERY == 0:
            record(step)
    x, _, _ = eager_metrics(theta, im["eps"], im["label"], im["noise_labels"], im["y"], im["hr"])
    result = {"image_index": i, "lr_theta": lr_theta, "steps": STEPS, "seconds": perf_counter() - start,
              "csgm_restart": im["restart"], "trace": trace}
    buffer = io.BytesIO()
    np.savez_compressed(buffer, result_json=json.dumps(result), x=x)
    atomic_write_bytes(path, buffer.getvalue())
    del theta, state
    return result, x


runs = {}
for lr_theta in LR_THETAS:
    opt, step_fn = make_theta_step(lr_theta)
    for i in DEV_IMAGES:
        result, x = run_theta_only(i, lr_theta, step_fn, opt)
        runs[(lr_theta, i)] = result
        images[i]["outputs"][f"CSGM + θ (lr {lr_theta:g})"] = x
        print(f"θ lr {lr_theta:g}, image {i}: {result['seconds']:.0f} s, "
              f"PSNR {psnr(images[i]['hr'], images[i]['outputs']['CSGM best']):.2f} → {psnr(images[i]['hr'], x):.2f} dB")

for i in DEV_IMAGES:
    im = images[i]
    im["metrics"] = {n: evaluate(x, im["hr"], im["y"]) for n, x in im["outputs"].items()}
METHODS = list(images[DEV_IMAGES[0]]["outputs"])

## Results 1: before vs after (eager, step 2,000; mean over 4 development images)

- **Δ CSGM** is the mean paired PSNR change from CSGM best; **> CSGM** counts images that improved.
- **LR loss** is the measurement fit (what the optimization minimizes); PSNR and band SNR measure HR quality.

In [ ]:
csgm_psnr = np.array([images[i]["metrics"]["CSGM best"]["psnr"] for i in DEV_IMAGES])
lines = ["| Method | PSNR | clipped | Δ CSGM | > CSGM | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |",
         "|---|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
for name in METHODS:
    ms = [images[i]["metrics"][name] for i in DEV_IMAGES]
    p = np.array([m["psnr"] for m in ms])
    lines.append(f"| {name} | {p.mean():.2f} | {np.mean([m['psnr_clipped'] for m in ms]):.2f} | "
                 f"{np.mean(p - csgm_psnr):+.2f} | {int(np.sum(p > csgm_psnr + 1e-9))}/4 | "
                 f"{np.mean([m['lr_loss'] for m in ms]):.2f} | "
                 + " | ".join(f"{np.mean([m['band_snr_db'][b] for m in ms]):.2f}" for b in BAND_NAMES) + " |")
display(Markdown("\n".join(lines)))

for key, title in (("psnr", "HR PSNR (dB)"), ("lr_loss", "LR loss")):
    lines = ["| Method | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " |",
             "|---|" + "---:|" * len(DEV_IMAGES)]
    for name in METHODS:
        lines.append(f"| {name} | " + " | ".join(f"{images[i]['metrics'][name][key]:.2f}" for i in DEV_IMAGES) + " |")
    display(Markdown(f"**{title} per image**\n\n" + "\n".join(lines)))

## Results 2: how quality changes over the 2,000 steps

Mean over the 4 images, every 50 steps (compiled path). Step 0 is CSGM best. Dashed lines: bicubic and bicubic + BP (eager). The table below gives, per learning rate, the HR-best step on the mean curve; **choosing that step uses HR, so it's an oracle diagnostic, not a method**.

In [ ]:
panels = [("lr_loss", "LR loss (log)"), ("psnr", "HR PSNR (dB)")] + [(b, f"SNR {b} (dB)") for b in BAND_NAMES]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for (key, title), ax in zip(panels, axes.flat):
    for k, lr_theta in enumerate(LR_THETAS):
        traces = [runs[(lr_theta, i)]["trace"] for i in DEV_IMAGES]
        ax.plot(traces[0]["step"], np.mean([t[key] for t in traces], axis=0), color=f"C{k}", label=f"θ lr {lr_theta:g}")
    if key != "lr_loss":
        for base, color in (("bicubic", "k"), ("bicubic + BP", "gray")):
            value = np.mean([images[i]["metrics"][base]["psnr"] if key == "psnr"
                             else images[i]["metrics"][base]["band_snr_db"][key] for i in DEV_IMAGES])
            ax.axhline(value, color=color, linestyle="--", linewidth=1, label=base)
    else:
        ax.set_yscale("log")
    ax.set(title=title, xlabel="θ-only step")
    ax.grid(alpha=0.3)
axes.flat[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

lines = ["| θ lr | PSNR step 0 | HR-best step (oracle) | PSNR there | PSNR step 2,000 | LR loss step 0 → 2,000 |",
         "|---:|---:|---:|---:|---:|---:|"]
for lr_theta in LR_THETAS:
    traces = [runs[(lr_theta, i)]["trace"] for i in DEV_IMAGES]
    mean_psnr = np.mean([t["psnr"] for t in traces], axis=0)
    mean_loss = np.mean([t["lr_loss"] for t in traces], axis=0)
    k = int(np.argmax(mean_psnr))
    lines.append(f"| {lr_theta:g} | {mean_psnr[0]:.2f} | {traces[0]['step'][k]} | {mean_psnr[k]:.2f} | "
                 f"{mean_psnr[-1]:.2f} | {mean_loss[0]:.1f} → {mean_loss[-1]:.2f} |")
display(Markdown("**Mean curve summary (compiled path)**\n\n" + "\n".join(lines)))

## Results 3: side by side

Top row: full images. Bottom row: the same 80×80 crop (marked in yellow on the original), enlarged to show detail. Titles give eager HR PSNR, LR loss and band SNR low / mid / high / very high.

In [ ]:
SHOW = ["bicubic", "CSGM best"] + [f"CSGM + θ (lr {lr:g})" for lr in LR_THETAS]
CROP = 80


def crop_corner(hr):
    # Crop with the most mid/high-frequency HR detail (gradient energy), chosen on HR only for display.
    g = np.abs(np.diff(hr.mean(-1), axis=0))[:, :-1] + np.abs(np.diff(hr.mean(-1), axis=1))[:-1, :]
    best, pos = -1, (0, 0)
    for r in range(0, 256 - CROP, 16):
        for c in range(0, 256 - CROP, 16):
            s = g[r:r + CROP, c:c + CROP].sum()
            if s > best:
                best, pos = s, (r, c)
    return pos


to_img = lambda x: (np.clip(x, -1, 1) + 1) / 2
for i in DEV_IMAGES:
    im = images[i]
    r, c = crop_corner(im["hr"])
    fig, axes = plt.subplots(2, len(SHOW) + 1, figsize=(4 * (len(SHOW) + 1), 8.6))
    axes[0, 0].imshow(to_img(im["hr"]))
    axes[0, 0].add_patch(plt.Rectangle((c, r), CROP, CROP, fill=False, color="yellow", linewidth=1.5))
    axes[0, 0].set_title(f"Original HR, development image {i}\n{class_name(im['label'])}", fontsize=9)
    axes[1, 0].imshow(to_img(im["hr"][r:r + CROP, c:c + CROP]), interpolation="nearest")
    axes[1, 0].set_title("Original HR (crop)", fontsize=9)
    for k, name in enumerate(SHOW, start=1):
        m, x = im["metrics"][name], im["outputs"][name]
        snr = "/".join(f"{m['band_snr_db'][b]:.1f}" for b in BAND_NAMES)
        axes[0, k].imshow(to_img(x))
        axes[0, k].set_title(f"{name}\nPSNR {m['psnr']:.2f} dB | LR loss {m['lr_loss']:.1f}\nSNR {snr}", fontsize=8)
        axes[1, k].imshow(to_img(x[r:r + CROP, c:c + CROP]), interpolation="nearest")
        axes[1, k].set_title(f"{name} (crop)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.9, wspace=0.04, hspace=0.2)
    plt.show()

## Conclusions (4 development images, correct label; small sample)

**Observations**

1. **θ-only adaptation lowers HR PSNR on every image at both learning rates.** After 2,000 steps (eager):
   - θ lr 1e-6: 21.06 → **18.78 dB** (−2.28 dB, 0/4 improve);
   - θ lr 1e-5: 21.06 → **18.18 dB** (−2.88 dB, 0/4 improve).
   - For comparison: bicubic 24.94, bicubic + BP 25.43.
2. **It does exactly what it optimizes: measurement fit improves a lot.** LR loss 50.5 → 7.8 (lr 1e-6) and → 1.1 (lr 1e-5).
3. **The trade-off is the same one seen in notebooks 02 and 04, now cleanly attributable to θ alone.**
   - Measured bands improve: low 20.7 → 28–34 dB, mid 5.4 → 9.7–11.3 dB.
   - Invented bands get worse: high −2.6 → −4.2 dB, very high −6.0 → −9.9 / −10.8 dB.
   - Even the improved low/mid bands stay below bicubic + BP (56.8 / 14.4 dB).
4. **Over the steps** (compiled-path curves): PSNR peaks very early and briefly (oracle-chosen step 50: +0.04 dB at lr 1e-6, +0.28 dB at lr 1e-5) and then declines. The lr 1e-5 run has essentially converged by about 1,000 steps; lr 1e-6 is still moving along the same path at 2,000 steps. Doubling the steps from 1,000 (notebook 02) to 2,000 makes lr 1e-6 worse (19.9 → 18.8 dB), not better.
5. **Results match notebook 02's ε-lr-0 runs at 1,000 steps** (lr 1e-5: 18.29 there vs 18.18 here), which confirms that ε contributes almost nothing and θ drives the adaptation.

**Visual impression** (side-by-side figures; subjective, not measured)
- CSGM + θ follows the true image's large-scale structure and colours more closely than CSGM best: shapes and edges line up better with the original, and CSGM's block artifacts are partly replaced.
- But it adds dense, high-contrast high-frequency texture (streaks, colour noise) that is not in the original. That texture is what PSNR and the high-band SNR penalize.

**Interpretation (hypothesis)**
- With ε fixed, θ has enough freedom (134M parameters) to fit y almost exactly. Nothing in the loss constrains frequencies above the LR limit, so they drift into artifacts. A regularizer on θ (weight anchor λ‖θ − θ₀‖²) or an early stop is the obvious thing to test; whether either helps on a *perceptual* metric is the open question.

**Implications**
- On PSNR, θ-only adaptation from CSGM is not a gain at any step count tested, except a ≤ 0.3 dB oracle-timed blip at step 50.
- If IADrift is to be argued for, it has to be on perceptual quality (LPIPS, visual examples), which needs the LPIPS install.